# Home Energy Management — Full Pipeline (Self-Contained)

**No external files needed.** Data loaded directly from GitHub CSV. All code inlined.

Targets: `target_load_kW_15`, `target_pv_kW_15`, `target_load_kW_1D`, `target_pv_kW_1D`

---
**Split strategy:** Per-house temporal split (70% train, 15% val, 15% test) — every house appears in all splits.

## 1. Install & Import

In [ ]:
!pip install -q xgboost tensorflow pyyaml

import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import xgboost as xgb

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
print('Ready.')

## 2. Config

In [ ]:
config = {
    'data': {
        'raw_url': 'https://raw.githubusercontent.com/ExplorerGumel/Home-Energy-Management/main/HEMS_dataset.csv',
        'timestamp_col': 'timestamp',
        'house_id_col': 'house_id',
        'date_format': '%d-%m-%Y %H:%M',
        'freq_minutes': 15,
    },
    'targets': {
        'load_15': 'target_load_kW_15',
        'pv_15': 'target_pv_kW_15',
        'load_1d': 'target_load_kW_1D',
        'pv_1d': 'target_pv_kW_1D',
    },
    'split': {
        'method': 'temporal_per_house',
        'train_frac': 0.70,
        'val_frac': 0.15,
        'test_frac': 0.15,
    },
    'feature_engineering': {
        'cyclical_encode': ['hour_of_day', 'day_of_week', 'minute_of_hour'],
        'lag_features': {
            'enabled': True,
            'columns': ['total_demand_kW', 'pv_generation_kW', 'outdoor_temp_C', 'solar_irradiance_Wm2', 'battery_soc_kWh'],
            'lags_15min': [1, 2, 4, 8, 12, 24],
            'lags_1d': [96],
        },
        'rolling_features': {
            'enabled': True,
            'windows_15min': [4, 8, 24, 96],
            'windows_1d': [96, 336],
            'columns': ['total_demand_kW', 'pv_generation_kW', 'outdoor_temp_C'],
        },
        'appliance_features': {
            'enabled': True,
            'time_since_window': 96,
        },
    },
    'model': {
        'architecture': 'multi_task',
        'sequence_length': 96,
        'forecast_15_steps': 1,
        'forecast_1d_steps': 96,
        'shared_encoder': {'type': 'lstm', 'units': 128, 'layers': 2, 'dropout': 0.2},
        'head_15min': {'units': [64, 32], 'dropout': 0.2},
        'head_1d': {'units': [64, 32], 'dropout': 0.2},
        'xgboost': {'n_estimators': 500, 'learning_rate': 0.05, 'max_depth': 7, 'subsample': 0.8, 'colsample_bytree': 0.8, 'early_stopping_rounds': 50},
    },
    'training': {
        'batch_size': 256, 'epochs': 100, 'learning_rate': 0.001, 'optimizer': 'adam',
        'early_stopping_patience': 10, 'reduce_lr_patience': 5, 'reduce_lr_factor': 0.5,
    },
}

## 3. Load Data

In [ ]:
print('Downloading dataset from GitHub (~34MB)...')
df = pd.read_csv(config['data']['raw_url'])
df['timestamp'] = pd.to_datetime(df['timestamp'], format=config['data']['date_format'])
df = df.sort_values(['house_id', 'timestamp']).reset_index(drop=True)

target_cols = list(config['targets'].values())
print(f"Houses: {df['house_id'].nunique()}, Rows: {len(df):,}")
print(f"Date range: {df['timestamp'].min()} to {df['timestamp'].max()}")
print(f"Targets: {target_cols}")

## 4. EDA

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, t in zip(axes.flatten()[:4], target_cols):
    ax.hist(df[t].dropna(), bins=80, edgecolor='black', alpha=0.7, color='steelblue')
    ax.set_title(t)
sns.heatmap(df[target_cols].corr(), annot=True, cmap='RdBu_r', center=0, ax=axes[1, 2])
axes[1, 2].set_title('Target Correlations')
plt.tight_layout(); plt.show()

In [ ]:
hh = df[df['house_id'] == df['house_id'].iloc[0]].iloc[:500]
plt.figure(figsize=(16, 3))
plt.plot(hh['timestamp'], hh['total_demand_kW'], label='Demand', linewidth=0.8)
plt.plot(hh['timestamp'], hh['pv_generation_kW'], label='PV', linewidth=0.8)
plt.legend(); plt.title('Load & PV (house 1, first 500 steps)')
plt.show()

## 5. Per-House Temporal Split

**Key insight:** Time series must be split chronologically per house. Using a fixed date boundary across all houses is wrong because houses may have different data start/end dates. We split each house individually: first 70% → train, next 15% → val, last 15% → test.

In [ ]:
split_cfg = config['split']

train_dfs, val_dfs, test_dfs = [], [], []

for hid, group in df.groupby('house_id'):
    group = group.sort_values('timestamp').reset_index(drop=True)
    n = len(group)
    t1 = int(n * split_cfg['train_frac'])
    t2 = t1 + int(n * split_cfg['val_frac'])
    train_dfs.append(group.iloc[:t1])
    val_dfs.append(group.iloc[t1:t2])
    test_dfs.append(group.iloc[t2:])

train_df = pd.concat(train_dfs).reset_index(drop=True)
val_df = pd.concat(val_dfs).reset_index(drop=True)
test_df = pd.concat(test_dfs).reset_index(drop=True)

print(f"Train: {len(train_df):,} rows ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df):,} rows ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df):,} rows ({len(test_df)/len(df)*100:.1f}%)")
print()

# Verify every house appears in all splits
for name, split_df in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    n_houses = split_df['house_id'].nunique()
    rng = f"{split_df['timestamp'].min()} → {split_df['timestamp'].max()}"
    print(f"{name:<6} | houses={n_houses} | {rng}")

In [ ]:
# Save split CSVs so you can inspect them
train_df.to_csv('train.csv', index=False)
val_df.to_csv('val.csv', index=False)
test_df.to_csv('test.csv', index=False)
print('Saved train.csv, val.csv, test.csv')

## 6. Feature Engineering

**Critical:** Fit feature engineering on train only to avoid data leakage.

In [ ]:
fe_cfg = config['feature_engineering']

def engineer_features(df_in, is_train=True, ref_df=None):
    df = df_in.copy()
    
    # --- cyclical ---
    for col in fe_cfg['cyclical_encode']:
        max_v = {'hour_of_day': 24, 'minute_of_hour': 60, 'day_of_week': 7}[col]
        df[f'{col}_sin'] = np.sin(2 * np.pi * df[col] / max_v)
        df[f'{col}_cos'] = np.cos(2 * np.pi * df[col] / max_v)
    
    # --- lags (fit on train, transform on val/test) ---
    if fe_cfg['lag_features']['enabled']:
        lag_cols = fe_cfg['lag_features']['columns']
        all_lags = fe_cfg['lag_features']['lags_15min'] + fe_cfg['lag_features']['lags_1d']
        source = ref_df if not is_train and ref_df is not None else df
        for col in lag_cols:
            for lag in all_lags:
                full_series = pd.concat([source[col], df[col]], axis=0) if not is_train else df[col]
                shifted = df.groupby('house_id')[col].shift(lag)
                df[f'{col}_lag_{lag}'] = shifted
    
    # --- rolling ---
    if fe_cfg['rolling_features']['enabled']:
        roll_cols = fe_cfg['rolling_features']['columns']
        all_windows = fe_cfg['rolling_features']['windows_15min'] + fe_cfg['rolling_features']['windows_1d']
        for col in roll_cols:
            for w in all_windows:
                grouped = df.groupby('house_id')[col]
                df[f'{col}_rm_{w}'] = grouped.transform(lambda x: x.shift(1).rolling(w, min_periods=1).mean())
                df[f'{col}_rs_{w}'] = grouped.transform(lambda x: x.shift(1).rolling(w, min_periods=1).std().fillna(0))
    
    # --- appliance ---
    if fe_cfg['appliance_features']['enabled']:
        window = fe_cfg['appliance_features']['time_since_window']
        for app in ['washing_machine_on', 'dishwasher_on', 'water_heater_on', 'ev_charging_on']:
            def steps_since(s):
                res = np.full(len(s), window, dtype=float)
                last = -1
                for i in range(len(s)):
                    if s.iloc[i] == 1: last = i
                    if last >= 0: res[i] = min(i - last, window)
                return res
            df[f'{app}_steps_since'] = df.groupby('house_id')[app].transform(steps_since)
    
    return df

print('Engineering train features...')
train_df = engineer_features(train_df, is_train=True)
print('Engineering val features (using train context)...')
val_df = engineer_features(val_df, is_train=False, ref_df=train_df)
print('Engineering test features (using train context)...')
test_df = engineer_features(test_df, is_train=False, ref_df=train_df)

feature_cols = [c for c in train_df.columns if c not in target_cols + ['timestamp', 'house_id']]
print(f'\nTotal features: {len(feature_cols)}')
null_pct = train_df[feature_cols].isnull().sum().sum() / (len(train_df) * len(feature_cols)) * 100
print(f'Train NaN %: {null_pct:.2f}% (from lag/rolling edges — will be dropped in sequencing)')

## 7. Build Sequences

Sliding window of 96 timesteps (24h). Rows with NaN from lag/rolling edges are dropped.

In [ ]:
seq_len = config['model']['sequence_length']

def make_sequences(df_sub):
    vals = df_sub[feature_cols].values
    tgts = {t: df_sub[t].values for t in target_cols}
    X, y = [], {t: [] for t in target_cols}
    for i in range(len(vals) - seq_len):
        x_seq = vals[i:i+seq_len]
        if np.any(np.isnan(x_seq)): continue
        X.append(x_seq)
        for t in target_cols: y[t].append(tgts[t][i+seq_len])
    X_arr = np.array(X)
    y_arr = {t: np.clip(np.array(y[t]).reshape(-1, 1), 0, None) for t in target_cols}
    return X_arr, y_arr

X_train, y_train = make_sequences(train_df)
X_val, y_val = make_sequences(val_df)
X_test, y_test = make_sequences(test_df)
print(f'X_train: {X_train.shape}  y: {list(y_train.values())[0].shape}')
print(f'X_val:   {X_val.shape}')
print(f'X_test:  {X_test.shape}')

## 8. Multi-Task LSTM

In [ ]:
md = config['model']
enc = md['shared_encoder']

inp = tf.keras.layers.Input(shape=(seq_len, X_train.shape[2]), name='input')
x = inp
for i in range(enc['layers']):
    x = tf.keras.layers.LSTM(
        enc['units'], return_sequences=(i < enc['layers']-1),
        dropout=enc['dropout'], name=f'lstm_{i}'
    )(x)

h15 = x
for u in md['head_15min']['units']:
    h15 = tf.keras.layers.Dense(u, activation='relu')(h15)
    h15 = tf.keras.layers.Dropout(md['head_15min']['dropout'])(h15)

h1d = x
for u in md['head_1d']['units']:
    h1d = tf.keras.layers.Dense(u, activation='relu')(h1d)
    h1d = tf.keras.layers.Dropout(md['head_1d']['dropout'])(h1d)

model = tf.keras.Model(inp, [
    tf.keras.layers.Dense(1, name=target_cols[0])(h15),
    tf.keras.layers.Dense(1, name=target_cols[1])(h15),
    tf.keras.layers.Dense(1, name=target_cols[2])(h1d),
    tf.keras.layers.Dense(1, name=target_cols[3])(h1d),
])

model.compile(
    optimizer='adam',
    loss={t: 'mse' for t in target_cols},
    loss_weights={t: 1.0 for t in target_cols[:2]} | {t: 0.5 for t in target_cols[2:]},
)
model.summary()

In [ ]:
tr = config['training']
history = model.fit(
    X_train, {t: y_train[t] for t in target_cols},
    epochs=tr['epochs'], batch_size=tr['batch_size'],
    validation_data=(X_val, {t: y_val[t] for t in target_cols}),
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=tr['early_stopping_patience'], restore_best_weights=True
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', patience=tr['reduce_lr_patience'], factor=tr['reduce_lr_factor']
        ),
    ],
    verbose=1,
)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(history.history['loss'], label='train_loss')
ax.plot(history.history['val_loss'], label='val_loss')
ax.legend(); ax.set_title('Training Loss')
plt.show()

## 9. Evaluate LSTM

In [ ]:
def compute_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred)**2))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else 0.0
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else 0.0
    return {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2}

preds = model.predict(X_test, verbose=0)
lstm_results = {}
fig, axes = plt.subplots(2, 2, figsize=(16, 7))
for i, t in enumerate(target_cols):
    yt = y_test[t].flatten()
    yp = preds[i].flatten()
    lstm_results[t] = compute_metrics(yt, yp)
    ax = axes.flatten()[i]
    n = min(300, len(yt))
    ax.plot(yt[:n], label='Actual', alpha=0.8, linewidth=0.7)
    ax.plot(yp[:n], label='Pred', alpha=0.8, linewidth=0.7, linestyle='--')
    r = lstm_results[t]
    ax.set_title(f'{t}\nMAE={r["mae"]:.4f}  RMSE={r["rmse"]:.4f}  R²={r["r2"]:.4f}')
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10. XGBoost Baseline

In [ ]:
X_train_f = X_train.reshape(X_train.shape[0], -1)
X_val_f = X_val.reshape(X_val.shape[0], -1)
X_test_f = X_test.reshape(X_test.shape[0], -1)
y_train_m = np.column_stack([y_train[t] for t in target_cols])
y_val_m = np.column_stack([y_val[t] for t in target_cols])
y_test_m = np.column_stack([y_test[t] for t in target_cols])

xgb_params = config['model']['xgboost']
xgb_results = {}
for i, t in enumerate(target_cols):
    print(f'Training {t}...')
    xgb_model = xgb.XGBRegressor(
        n_estimators=xgb_params['n_estimators'], learning_rate=xgb_params['learning_rate'],
        max_depth=xgb_params['max_depth'], subsample=xgb_params['subsample'],
        colsample_bytree=xgb_params['colsample_bytree'],
        early_stopping_rounds=xgb_params['early_stopping_rounds'],
        random_state=42, verbosity=0
    )
    xgb_model.fit(X_train_f, y_train_m[:, i], eval_set=[(X_val_f, y_val_m[:, i])], verbose=False)
    preds = xgb_model.predict(X_test_f)
    xgb_results[t] = compute_metrics(y_test_m[:, i], preds)

In [ ]:
print(f"{'Target':<25} {'Model':<10} {'MAE':<10} {'RMSE':<10} {'R²':<10} {'MAPE(%)':<8}")
print('-' * 63)
for t in target_cols:
    for model_name, res in [('LSTM', lstm_results), ('XGBoost', xgb_results)]:
        r = res[t]
        print(f"{t:<25} {model_name:<10} {r['mae']:<10.4f} {r['rmse']:<10.4f} {r['r2']:<10.4f} {r['mape']:<8.2f}")
    print()

## 11. Save Model + Download

In [ ]:
model.save('hems_model.h5')
print('Model saved as hems_model.h5')

from google.colab import files
files.download('hems_model.h5')